In [9]:
# Pega os lembretes novos na planilha

import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

scope = [
    'https://spreadsheets.google.com/feeds',
    'https://www.googleapis.com/auth/drive'
]

creds = ServiceAccountCredentials.from_json_keyfile_name('secret.json', scope)
client = gspread.authorize(creds)

spreadsheet = client.open_by_key("1gjz0hk8dL0rbqzTfHGLbzwXnA-h8AVxZtgffCbseDIk")
sheet = spreadsheet.sheet1

data = sheet.get_all_records()
df = pd.DataFrame(data)
df = df.loc[:, (df != '').any(axis=0)]

df.rename(columns={"Tipo de Pedido": "tipo_pedido"}, inplace=True)
df["num_processo"] = df["num_processo"].str.strip()
if 'tipo_pedido' not in df.columns and 'tipo de pedido' in df.columns:
    df.rename(columns={'tipo de pedido': 'tipo_pedido'}, inplace=True)

s = df['tipo_pedido']
mask = s.notnull() & (s.astype(str).str.strip() != '')
df = df[mask].copy()

df = df[['unidade', 'num_processo', 'tipo_pedido']].copy()
df

,unidade,num_processo,tipo_pedido
0,SRD1CIV,5004629-20.2024.8.21.0069,@1184
1,SRD1CIV,5001789-42.2021.8.21.0069,@1184
2,SRD1CIV,5000457-79.2017.8.21.0069,@1184
3,SRD1CIV,5004583-31.2024.8.21.0069,@acordofiscal
4,SRD1CIV,5004410-41.2023.8.21.0069,@acordofiscal
5,SRD1CIV,5004319-14.2024.8.21.0069,@acordofiscal
6,SRD1CIV,5004253-34.2024.8.21.0069,@acordofiscal
7,SRD1CIV,5004229-06.2024.8.21.0069,@acordofiscal
8,SRD1CIV,5004117-37.2024.8.21.0069,@acordofiscal
9,SRD1CIV,5003867-04.2024.8.21.0069,@acordofiscal


In [10]:
df


,unidade,num_processo,tipo_pedido
0,SRD1CIV,5004629-20.2024.8.21.0069,@1184
1,SRD1CIV,5001789-42.2021.8.21.0069,@1184
2,SRD1CIV,5000457-79.2017.8.21.0069,@1184
3,SRD1CIV,5004583-31.2024.8.21.0069,@acordofiscal
4,SRD1CIV,5004410-41.2023.8.21.0069,@acordofiscal
5,SRD1CIV,5004319-14.2024.8.21.0069,@acordofiscal
6,SRD1CIV,5004253-34.2024.8.21.0069,@acordofiscal
7,SRD1CIV,5004229-06.2024.8.21.0069,@acordofiscal
8,SRD1CIV,5004117-37.2024.8.21.0069,@acordofiscal
9,SRD1CIV,5003867-04.2024.8.21.0069,@acordofiscal


In [11]:
# Reseta a tabela

import sqlite3
db = "urcaciv.db"
conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()

cursor.execute(f"DELETE FROM lembretes_to_do")
conn.commit()

conn.close()

print("Todos os registros da tabela 'lembretes_to_do' foram apagados.")

Todos os registros da tabela 'lembretes_to_do' foram apagados.


In [12]:
from datetime import datetime

df["created_at"] = datetime.now().strftime("%d/%m/%Y")
df["dt_lembrete"] = None

print(df)
df.to_sql("lembretes_to_do", "sqlite:///urcaciv.db", if_exists="append", index=False)

    unidade               num_processo              tipo_pedido  created_at  \
0   SRD1CIV  5004629-20.2024.8.21.0069                    @1184  11/11/2025   
1   SRD1CIV  5001789-42.2021.8.21.0069                    @1184  11/11/2025   
2   SRD1CIV  5000457-79.2017.8.21.0069                    @1184  11/11/2025   
3   SRD1CIV  5004583-31.2024.8.21.0069            @acordofiscal  11/11/2025   
4   SRD1CIV  5004410-41.2023.8.21.0069            @acordofiscal  11/11/2025   
5   SRD1CIV  5004319-14.2024.8.21.0069            @acordofiscal  11/11/2025   
6   SRD1CIV  5004253-34.2024.8.21.0069            @acordofiscal  11/11/2025   
7   SRD1CIV  5004229-06.2024.8.21.0069            @acordofiscal  11/11/2025   
8   SRD1CIV  5004117-37.2024.8.21.0069            @acordofiscal  11/11/2025   
9   SRD1CIV  5003867-04.2024.8.21.0069            @acordofiscal  11/11/2025   
10  SRD1CIV  5003375-46.2023.8.21.0069            @acordofiscal  11/11/2025   
11  SRD1CIV  5003351-52.2022.8.21.0069            @a

54